# Notebook 0: Data Preparation

Prepare cybersecurity news training data for fine-tuning summarization models.

**Sources:**
- Upload your own JSONL data
- Scrape from RSS/API feeds
- Use HuggingFace datasets
- Augment with synthetic summaries

In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/model-lab/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"\nExpected format per line (JSONL):")
print(json.dumps({"article": "...", "summary": "..."}, indent=2))

## Option 1: Upload Local File

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Validate and convert to JSONL
data = []
with open(filename) as f:
    content = f.read().strip()
    
    # Try JSONL first
    try:
        for line in content.split('\n'):
            if line.strip():
                row = json.loads(line)
                assert "article" in row and "summary" in row
                data.append(row)
    except (json.JSONDecodeError, AssertionError):
        # Try JSON array
        try:
            rows = json.loads(content)
            if isinstance(rows, list):
                for row in rows:
                    assert "article" in row and "summary" in row
                    data.append(row)
            else:
                data.append(rows)
        except:
            print("Error: File must be JSONL or JSON array with 'article' and 'summary' fields")
            raise

print(f"Loaded {len(data)} samples")
print(f"\nSample article (first 100 chars): {data[0]['article'][:100]}...")
print(f"Sample summary: {data[0]['summary'][:100]}...")

# Save to Drive
with open(DATA_DIR / "train.jsonl", "w") as f:
    for row in data:
        f.write(json.dumps(row) + "\n")
print(f"\nSaved to {DATA_DIR / 'train.jsonl'}")

## Option 2: HuggingFace Datasets

Popular summarization datasets that work well for cybersecurity fine-tuning.

In [ ]:
from datasets import load_dataset

# CNN/DailyMail - general news summarization baseline
cnn_dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1000]")
cnn_data = [{"article": r["article"], "summary": r["highlights"]} for r in cnn_dataset]

with open(DATA_DIR / "cnn_dailymail_1k.jsonl", "w") as f:
    for row in cnn_data:
        f.write(json.dumps(row) + "\n")

print(f"Saved {len(cnn_data)} CNN/DailyMail samples")
print(f"\nSample:\n  Article: {cnn_data[0]['article'][:100]}...")
print(f"  Summary: {cnn_data[0]['summary'][:100]}...")

## Option 3: Augment with Synthetic Summaries

Use an existing model to generate summaries for raw articles, then human-review them.

In [ ]:
from transformers import pipeline

# Load raw articles (no summaries yet)
RAW_ARTICLES = [
    "Kaspersky researchers discovered a new advanced persistent threat group targeting diplomatic organizations in the Middle East using compromised USB drives to jump air gaps.",
    "The European Union Agency for Cybersecurity released its annual report showing supply chain compromises increased 180% in 2023, with software supply chains as the primary attack vector.",
]

summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device_map="auto")

augmented_data = []
for article in RAW_ARTICLES:
    result = summarizer(article, max_length=100, min_length=20, do_sample=False)
    summary = result[0]["summary_text"]
    augmented_data.append({"article": article, "summary": summary})
    print(f"Article: {article[:80]}...")
    print(f"Generated summary: {summary}")
    print()

print(f"\nGenerated {len(augmented_data)} synthetic summaries.")
print("IMPORTANT: Review and edit these summaries before using for training!")

## Train/Val/Test Split

In [ ]:
import random

# Load your data
input_file = DATA_DIR / "train.jsonl"  # Change as needed
with open(input_file) as f:
    all_data = [json.loads(line) for line in f if line.strip()]

random.seed(42)
random.shuffle(all_data)

train_end = int(len(all_data) * 0.8)
val_end = int(len(all_data) * 0.9)

train_data = all_data[:train_end]
val_data = all_data[train_end:val_end]
test_data = all_data[val_end:]

for split_name, split_data, filename in [
    ("train", train_data, "train.jsonl"),
    ("validation", val_data, "validation.jsonl"),
    ("test", test_data, "test.jsonl"),
]:
    path = DATA_DIR / filename
    with open(path, "w") as f:
        for row in split_data:
            f.write(json.dumps(row) + "\n")
    print(f"{split_name}: {len(split_data)} samples -> {path}")

print(f"\nDone! Files saved in {DATA_DIR}")

## Data Quality Check

In [ ]:
import statistics

for split_name in ["train", "validation", "test"]:
    path = DATA_DIR / f"{split_name}.jsonl"
    if not path.exists():
        continue
    
    with open(path) as f:
        rows = [json.loads(line) for line in f if line.strip()]
    
    art_lens = [len(r["article"].split()) for r in rows]
    sum_lens = [len(r["summary"].split()) for r in rows]
    ratios = [s/a if a > 0 else 0 for s, a in zip(sum_lens, art_lens)]
    
    print(f"\n{split_name.upper()} ({len(rows)} samples):")
    print(f"  Article length: min={min(art_lens)}, max={max(art_lens)}, avg={statistics.mean(art_lens):.0f}")
    print(f"  Summary length: min={min(sum_lens)}, max={max(sum_lens)}, avg={statistics.mean(sum_lens):.0f}")
    print(f"  Compression ratio: avg={statistics.mean(ratios):.2f}")
    
    # Check for empty or very short entries
    bad = [i for i, r in enumerate(rows) if len(r["article"].split()) < 10 or len(r["summary"].split()) < 5]
    if bad:
        print(f"  WARNING: {len(bad)} entries with very short article or summary")